<a href="https://colab.research.google.com/github/tayyabp0g/w3_1FlyRank/blob/main/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Lane:** Refresh / Content Opportunity Scoring — *which pages show early signs of decline and deserve a human review first?*
**Slice:** `fact_content_daily_performance`, partition `month=2026-03` (a mid-panel month). The final month (June 2026) and the `_sample` table are **never touched** — June stays a sealed test month.

> Token safety: the Hugging Face READ token is read from the Colab Secret `HF_TOKEN` (or an env var). It is never typed into a cell.

In [20]:
%pip -q install duckdb huggingface_hub scikit-learn pandas pyarrow

In [21]:
# Step 1: Verification cell to check HF_TOKEN configuration
import os
from google.colab import userdata

try:
    token = userdata.get('HF_TOKEN')
    if token:
        print("✅ Success: HF_TOKEN detected in Colab Secrets!")
        print(f"Token starts with: {token[:8]}...")
    else:
        print("❌ Error: HF_TOKEN value is empty or not found in Secrets. Please add it in the 🔑 panel.")
except Exception as e:
    print(f"❌ Error accessing Colab Secrets: {e}\nMake sure to toggle 'Notebook access' to ON for HF_TOKEN.")

✅ Success: HF_TOKEN detected in Colab Secrets!
Token starts with: hf_aVnhX...


## 1. Unit of analysis + time window

**Contract answers 1–4 (plain words):**

1. **What one row means.** One row = **one content item** (one `client_hash_id` × `content_hash_id` pair) with its search numbers added up over a window. The source table itself is finer: one row per `report_date` × client × content. I check that grain in query A below.
2. **Which table(s).** Only `fact_content_daily_performance`, only the partition `month=2026-03`. No joins — everything I need is already in the daily fact table.
3. **Which time window.** The decision moment is the **end of 15 March 2026**. Features use 1–15 March only. The outcome window is 16–31 March. A mid-panel month on purpose: June 2026 is the sealed test month.
4. **What I predict (label / proxy).** `is_declining` = 1 when a page's **daily-average impressions in 16–31 March fell more than 20% below its daily average in 1–15 March**. It is a proxy for "this page is losing visibility and is worth a review", not proof of a cause. I keep only pages with at least 100 impressions in 1–15 March so tiny pages do not add noise. The output is a ranked review list (decision-support), not an automatic action.

In [22]:
import os, getpass
import duckdb, numpy as np, pandas as pd
from google.colab import userdata

# Securely load the configured token from Colab Secrets
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN')

# Fallback prompt if not configured
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

# Sanitize in case of accidental duplicate paste
if HF_TOKEN and len(HF_TOKEN) > 40 and HF_TOKEN.count('hf_') > 1:
    # split by 'hf_' and take the second part to keep it clean
    parts = HF_TOKEN.split('hf_')
    HF_TOKEN = 'hf_' + parts[-1].strip()

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL   = 'hf://datasets/FlyRank/internship-warehouse'
MONTH = '2026-03'                                   # mid-panel month (NOT the sealed June 2026)
FACT  = f"read_parquet('{REL}/fact_content_daily_performance/month={MONTH}/*.parquet')"

# --- the contract, as constants ---
H1_START, H1_END   = '2026-03-01', '2026-03-15'     # feature window (knowable at the decision moment)
H2_START, H2_END   = '2026-03-16', '2026-03-31'     # outcome window (the future we predict)
H1_DAYS, H2_DAYS   = 15, 16
MIN_IMP_H1         = 100                            # minimum-volume filter
DECLINE_FACTOR     = 0.8                            # decline = daily-avg impressions dropped by more than 20%

print('Table      : fact_content_daily_performance, month=' + MONTH)
print(f'Features   : {H1_START} -> {H1_END}   (decision moment = end of {H1_END})')
print(f'Outcome    : {H2_START} -> {H2_END}')
print(f'Label      : is_declining = daily-avg impressions in H2 < {DECLINE_FACTOR} x daily-avg in H1, only if imp_h1 >= {MIN_IMP_H1}')

Table      : fact_content_daily_performance, month=2026-03
Features   : 2026-03-01 -> 2026-03-15   (decision moment = end of 2026-03-15)
Outcome    : 2026-03-16 -> 2026-03-31
Label      : is_declining = daily-avg impressions in H2 < 0.8 x daily-avg in H1, only if imp_h1 >= 100


## 2. Fields: feature / label / context / excluded

**Contract answer 5 — one thing I deliberately exclude:** **every column dated after 15 March** (the whole 16–31 March window) is excluded from the features. It is the future relative to my decision moment; it is only allowed to build the label.

Other fields, sorted into the four buckets:

| Bucket | Fields | Why |
|---|---|---|
| **Feature** | `imp_h1`, `clicks_h1`, `ctr_h1`, `avg_pos_h1`, `active_days_h1` | Built from 1–15 March only |
| **Label / proxy** | `is_declining` (built from `imp_h2` vs `imp_h1`) | The outcome, never a feature |
| **Context** | `client_hash_id`, `content_hash_id` | Used to group and to split by client; pseudonyms, never features |
| **Excluded** | all 16–31 March columns; GA4 columns | Future information; GA4 values are zero-filled before a client's GA4 start (`ga4_data_available = FALSE`), so zeros there would be fake "no engagement" |

In [23]:
fields = pd.DataFrame([
    ('imp_h1',           'feature', 'sum of gsc_impressions, 1-15 Mar'),
    ('clicks_h1',        'feature', 'sum of gsc_clicks, 1-15 Mar'),
    ('ctr_h1',           'feature', 'clicks_h1 / imp_h1 (fraction, not x100)'),
    ('avg_pos_h1',       'feature', 'mean gsc_avg_position, 1-15 Mar (0 treated as missing)'),
    ('active_days_h1',   'feature', 'days with impressions > 0, 1-15 Mar'),
    ('is_declining',     'label',   'daily-avg impressions 16-31 Mar < 0.8 x daily-avg 1-15 Mar'),
    ('client_hash_id',   'context', 'grouping / split key, never a feature'),
    ('content_hash_id',  'context', 'item id, never a feature'),
    ('anything 16-31 Mar (except via the label)', 'excluded', 'future relative to the decision moment'),
    ('ga4_* columns',    'excluded', 'zero-filled before GA4 start; flag ga4_data_available checked in query C'),
], columns=['field', 'bucket', 'note'])
fields

,field,bucket,note
0,imp_h1,feature,"sum of gsc_impressions, 1-15 Mar"
1,clicks_h1,feature,"sum of gsc_clicks, 1-15 Mar"
2,ctr_h1,feature,"clicks_h1 / imp_h1 (fraction, not x100)"
3,avg_pos_h1,feature,"mean gsc_avg_position, 1-15 Mar (0 treated as ..."
4,active_days_h1,feature,"days with impressions > 0, 1-15 Mar"
5,is_declining,label,daily-avg impressions 16-31 Mar < 0.8 x daily-...
6,client_hash_id,context,"grouping / split key, never a feature"
7,content_hash_id,context,"item id, never a feature"
8,anything 16-31 Mar (except via the label),excluded,future relative to the decision moment
9,ga4_* columns,excluded,zero-filled before GA4 start; flag ga4_data_av...


## 3. Verify it with queries (grain, counts, missing values, windows)

Three small queries on `month=2026-03`. Each has a one-line "what I expect" so a surprise is visible.

### Query A — grain: is one row really `report_date` × client × content?
*Expect:* `duplicate_rows = 0` (every key appears once).

In [24]:
qa = con.sql(f"""
    SELECT COUNT(*)                                                         AS rows_total,
           COUNT(DISTINCT (report_date, client_hash_id, content_hash_id))  AS distinct_keys,
           COUNT(*) - COUNT(DISTINCT (report_date, client_hash_id, content_hash_id)) AS duplicate_rows
    FROM {FACT}
""").df()
display(qa)
print('GRAIN HOLDS' if int(qa.loc[0, 'duplicate_rows']) == 0 else 'GRAIN BROKEN - re-read the contract')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rows_total,distinct_keys,duplicate_rows
0,9841378,9841378,0


GRAIN HOLDS


### Query B — my slice: row count and date span
*Expect:* dates from `2026-03-01` to `2026-03-31`, with the row, client and content-item counts for this slice.

In [25]:
qb = con.sql(f"""
    SELECT COUNT(*)                         AS rows_in_slice,
           MIN(report_date)                 AS first_date,
           MAX(report_date)                 AS last_date,
           COUNT(DISTINCT report_date)      AS distinct_days,
           COUNT(DISTINCT client_hash_id)   AS clients,
           COUNT(DISTINCT content_hash_id)  AS content_items
    FROM {FACT}
""").df()
display(qb)

,rows_in_slice,first_date,last_date,distinct_days,clients,content_items
0,9841378,2026-03-01,2026-03-31,31,55,331437


### Query C — availability: filter with `IS TRUE`
`ga4_data_available` can be `FALSE` (zero-filled GA4) or even `NULL`. `= TRUE` silently drops NULLs in a way people forget; `IS TRUE` is explicit. The table shows how many rows survive. My five features are search-only, so I do **not** need GA4 and I do not drop rows for it — but the count tells me what any GA4 feature would have cost me.

In [26]:
qc = con.sql(f"""
    SELECT COUNT(*)                                          AS rows_total,
           COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS rows_ga4_available,
           ROUND(100.0 * COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) / COUNT(*), 2) AS pct_ga4_available,
           COUNT(*) FILTER (WHERE gsc_impressions > 0)        AS rows_with_gsc_impressions
    FROM {FACT}
""").df()
display(qc)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rows_total,rows_ga4_available,pct_ga4_available,rows_with_gsc_impressions
0,9841378,413966,4.21,3611061


### Five features (max) — one frame from the same month
Built inside DuckDB (one row per client × content), then pulled into pandas. Every feature uses **only 1–15 March**. The label uses 16–31 March.

| Feature | Knowable at the decision moment (end of 15 March) because… |
|---|---|
| `imp_h1` | …it is the sum of impressions on days 1–15, all already reported by the end of 15 March. |
| `clicks_h1` | …it is the sum of clicks on days 1–15, already reported by the decision moment. |
| `ctr_h1` | …it is clicks divided by impressions, both from days 1–15 only. |
| `avg_pos_h1` | …it is the mean search position on days 1–15; no later day enters the average. |
| `active_days_h1` | …it counts how many of days 1–15 had impressions; it cannot see anything after 15 March. |

In [27]:
frame = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(COALESCE(gsc_impressions, 0)) FILTER (WHERE report_date <= DATE '{H1_END}')   AS imp_h1,
           SUM(COALESCE(gsc_clicks, 0))      FILTER (WHERE report_date <= DATE '{H1_END}')   AS clicks_h1,
           AVG(NULLIF(gsc_avg_position, 0))  FILTER (WHERE report_date <= DATE '{H1_END}')   AS avg_pos_h1,
           COUNT(*)                          FILTER (WHERE report_date <= DATE '{H1_END}' AND gsc_impressions > 0) AS active_days_h1,
           SUM(COALESCE(gsc_impressions, 0)) FILTER (WHERE report_date >= DATE '{H2_START}') AS imp_h2   -- label ingredient, NOT a feature
    FROM {FACT}
    GROUP BY client_hash_id, content_hash_id
    HAVING SUM(COALESCE(gsc_impressions, 0)) FILTER (WHERE report_date <= DATE '{H1_END}') >= {MIN_IMP_H1}
""").df()

frame['imp_h2']   = frame['imp_h2'].fillna(0)
frame['ctr_h1']   = frame['clicks_h1'] / frame['imp_h1']          # imp_h1 >= 100, so no divide-by-zero
frame['is_declining'] = ((frame['imp_h2'] / H2_DAYS) < DECLINE_FACTOR * (frame['imp_h1'] / H1_DAYS)).astype(int)

FEATURES = ['imp_h1', 'clicks_h1', 'ctr_h1', 'avg_pos_h1', 'active_days_h1']   # exactly five
print(f'{len(frame):,} content items kept (imp_h1 >= {MIN_IMP_H1}); decline rate = {frame.is_declining.mean():.3f}')
frame[['client_hash_id', 'content_hash_id'] + FEATURES + ['is_declining']].head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

77,540 content items kept (imp_h1 >= 100); decline rate = 0.328


,client_hash_id,content_hash_id,imp_h1,clicks_h1,ctr_h1,avg_pos_h1,active_days_h1,is_declining
0,client_62f4a7e64f5e0096,content_d0dff76c889de68f,111.0,0.0,0.000000,5.222776,13,1
1,client_62f4a7e64f5e0096,content_2e6360ad20fd7107,219.0,1.0,0.004566,4.004356,15,0
2,client_62f4a7e64f5e0096,content_65c50dfe9d87a585,1494.0,0.0,0.000000,6.156643,14,0
3,client_62f4a7e64f5e0096,content_d49a012dcb924e31,246.0,0.0,0.000000,4.520919,15,1
4,client_62f4a7e64f5e0096,content_614baf2af4330bd7,413.0,1.0,0.002421,4.390322,15,0


### The honest score
Quick model, split **by client** (whole clients held out), so it is tested on clients it never saw. `avg_pos_h1` can be missing and I leave it missing (the model handles it natively) instead of filling with 0, which would fake a rank.

In [28]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score, average_precision_score

gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
tr_idx, te_idx = next(gss.split(frame, frame['is_declining'], groups=frame['client_hash_id']))
train, test = frame.iloc[tr_idx], frame.iloc[te_idx]

def quick_score(cols):
    m = HistGradientBoostingClassifier(max_iter=150, random_state=42).fit(train[cols], train['is_declining'])
    p = m.predict_proba(test[cols])[:, 1]
    return roc_auc_score(test['is_declining'], p), average_precision_score(test['is_declining'], p)

honest_auc, honest_ap = quick_score(FEATURES)
base_rate = test['is_declining'].mean()
print(f'test clients: {test.client_hash_id.nunique()}  | test rows: {len(test):,}  | base rate of decline: {base_rate:.3f}')
print(f'HONEST   ROC-AUC = {honest_auc:.3f}   avg precision = {honest_ap:.3f}   (random guessing: AUC 0.5, AP ~ {base_rate:.3f})')

test clients: 12  | test rows: 17,013  | base rate of decline: 0.201
HONEST   ROC-AUC = 0.520   avg precision = 0.210   (random guessing: AUC 0.5, AP ~ 0.201)


### The trap — add ONE label-derived column on purpose
`h2_vs_h1_ratio` is the 16–31 March daily average divided by the 1–15 March daily average — the label's own formula, just without the threshold. At the decision moment nobody could know it. Watch the score jump toward perfect, then delete it.

In [29]:
frame['h2_vs_h1_ratio'] = (frame['imp_h2'] / H2_DAYS) / (frame['imp_h1'] / H1_DAYS)   # LEAK: built from the future window
train, test = frame.iloc[tr_idx], frame.iloc[te_idx]

leaky_auc, leaky_ap = quick_score(FEATURES + ['h2_vs_h1_ratio'])
print(f'WITH THE LEAK  ROC-AUC = {leaky_auc:.3f}   avg precision = {leaky_ap:.3f}   <- suspiciously close to perfect')
print(f'jump vs honest: AUC +{leaky_auc - honest_auc:.3f}')

WITH THE LEAK  ROC-AUC = 1.000   avg precision = 1.000   <- suspiciously close to perfect
jump vs honest: AUC +0.480


### Delete the leak, keep the honest number
The leaky column is dropped from the frame. The number I keep and report is the **honest** one from the five features above. A near-perfect score from a model with a future-derived column is not skill — it is the answer key handed to the model.

In [30]:
frame = frame.drop(columns=['h2_vs_h1_ratio'])
train, test = frame.iloc[tr_idx], frame.iloc[te_idx]
assert 'h2_vs_h1_ratio' not in frame.columns

final_auc, final_ap = quick_score(FEATURES)
summary = pd.DataFrame({
    'run': ['honest (5 features)', 'with leak (6 columns)', 'FINAL - leak deleted (5 features)'],
    'ROC-AUC': [honest_auc, leaky_auc, final_auc],
    'avg precision': [honest_ap, leaky_ap, final_ap],
}).round(3)
display(summary)
print(f'Honest number I keep: ROC-AUC = {final_auc:.3f}')

,run,ROC-AUC,avg precision
0,honest (5 features),0.52,0.21
1,with leak (6 columns),1.00,1.00
2,FINAL - leak deleted (5 features),0.52,0.21


Honest number I keep: ROC-AUC = 0.520


## 4. Data limits

**One named limitation of my slice: it is a single month (March 2026) taken from an *unbalanced panel*.** Only clients that were already tracking in March appear, and a few large clients can dominate the rows, so the score describes *this slice*, not "FlyRank content in general". On top of that, the label compares two short windows (15 days vs 16 days) in one calendar month, so seasonality and ordinary day-to-day noise can look like "decline". These results are observational and directional — they support a review queue; they do not prove that any edit would fix a page.

In [31]:
per_client = frame.groupby('client_hash_id').size().sort_values(ascending=False)
print(f'clients in the slice : {per_client.size}')
print(f'largest client holds : {per_client.iloc[0] / per_client.sum():.1%} of kept content items')
print(f'top 3 clients hold   : {per_client.iloc[:3].sum() / per_client.sum():.1%} of kept content items')

clients in the slice : 38
largest client holds : 22.7% of kept content items
top 3 clients hold   : 55.0% of kept content items


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [32]:
# Automatic self-check (prints PASS/FAIL - nothing here changes your results)
import json, re
checks = {
    'exactly 3 verification queries (A, B, C) ran':          all(v in globals() for v in ['qa', 'qb', 'qc']),
    'grain holds (no duplicate report_date x client x content)': int(qa.loc[0, 'duplicate_rows']) == 0,
    'slice stays inside March 2026':                          str(qb.loc[0, 'first_date'])[:7] == MONTH and str(qb.loc[0, 'last_date'])[:7] == MONTH,
    'availability checked with IS TRUE':                      'rows_ga4_available' in qc.columns,
    'exactly five features':                                  len(FEATURES) == 5,
    'leak made the score jump':                               leaky_auc > honest_auc,
    'leak column deleted from the frame':                     'h2_vs_h1_ratio' not in frame.columns,
    'no token pasted in this notebook':                       True,
}
for name, ok in checks.items():
    print(('PASS  ' if ok else 'FAIL  ') + name)

PASS  exactly 3 verification queries (A, B, C) ran
PASS  grain holds (no duplicate report_date x client x content)
PASS  slice stays inside March 2026
PASS  availability checked with IS TRUE
PASS  exactly five features
PASS  leak made the score jump
PASS  leak column deleted from the frame
PASS  no token pasted in this notebook
